In [ ]:
# 导包
import time
import numpy as np
import pandas as pd
from google.protobuf.timestamp import to_datetime
from pandas import read_excel
from sqlalchemy import create_engine


from pyecharts.charts import Bar3D              # 需要安装下，即：pip install pyecharts
from pyecharts.commons.utils import JsCode
import pyecharts.options as opts

# 1. 加载数据

In [ ]:
# 1.1 定义列表：记录 excel 表名
sheet_names = ['2015', '2016', '2017', '2018', '会员等级']
# 1.2 从excel文件中读取数据，读取到：字典形式 ——> {‘2015’ : df对象, '2016' : df对象......}
# 参数1：文件名（路径）
# 参数2：excel表名
sheet_dict = pd.read_excel('./数据集/sales.xlsx', sheet_name = sheet_names)

In [ ]:
sheet_dict

In [ ]:
# 1.3 查看sheet_dict变量的数据类型类型
type(sheet_dict)        # <class 'dict'>

# 1.4 查看 2015 excel表的 dataframe对象
sheet_dict['2015']

# 1.5 查看 2015 excel表的 dataframe对象的基本信息
sheet_dict['2015'].info()

# 1.6 查看 2015 excel表的 dataframe对象的基本统计信息
sheet_dict['2015'].describe()

In [ ]:
# 1.7 查看 字典中每个 dataframe对象（即：每个sheet表）的基本信息 和 统计信息
for i in sheet_names:
    print(i)
    print(sheet_dict[i].info())
    print(sheet_dict[i].describe())

# 2. 数据预处理

In [ ]:
# 需要处理的动作：1. 删除缺失值， 2. 过滤出 消费金额 >1 的订单， 3. 固定时间节点，以每年的最后一天 作为当年的分析节点
# 1. 遍历获取到每张表（除了最后一张）
for i in sheet_names[:-1]:
    print(i)
    # 2. 删除缺失值
    sheet_dict[i].dropna(inplace = True)

    # 3. 过滤出 消费金额 >1 的订单
    # sheet_dict[i] = sheet_dict[i].query('"订单金额" > 1')
    sheet_dict[i] = sheet_dict[i][sheet_dict[i]['订单金额'] > 1]

    # 4. 固定时间节点，以每年的最后一天 作为当年的分析节点
    # sheet_dict[i]['max_year_date'] = to_datetime(f'{i}-12-31')
    sheet_dict[i]['max_year_date'] = sheet_dict[i]['提交日期'].max()

# 5. 查看处理后的数据
for i in sheet_names:
    print(i)
    print(sheet_dict[i].info())
    print(sheet_dict[i].describe())

In [24]:
# 把上面的四张表（四个dataframe对象） 合并到 到一张表上
# type(sheet_dict.values())
# type(list(sheet_dict.values()))
# type(list(sheet_dict.values())[:-1])
df_merge = pd.concat(list(sheet_dict.values())[:-1], ignore_index = True)       # 合并 并重置索引
df_merge

,会员ID,订单号,提交日期,订单金额,max_year_date
0,15278002468,3000304681,2015-01-01,499.0,2015-12-31
1,39236378972,3000305791,2015-01-01,2588.0,2015-12-31
2,38722039578,3000641787,2015-01-01,498.0,2015-12-31
3,11049640063,3000798913,2015-01-01,1572.0,2015-12-31
4,35038752292,3000821546,2015-01-01,10.1,2015-12-31
...,...,...,...,...,...
202822,39229485704,4354225182,2018-12-31,249.0,2018-12-31
202823,39229021075,4354225188,2018-12-31,89.0,2018-12-31
202824,39288976750,4354230034,2018-12-31,48.5,2018-12-31
202825,26772630,4354230163,2018-12-31,3196.0,2018-12-31
